In [2]:
import mlflow
import mlflow.xgboost
import numpy as np
import optuna
import os
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

In [3]:
train_df = pd.read_csv(os.path.join('..', 'data', 'processed', 'train_feature_engineered.csv'))
test_df = pd.read_csv(os.path.join('..', 'data', 'processed', 'test_feature_engineered.csv'))

In [4]:
target = "Carbon dioxide (CO2) emissions (total) excluding LULUCF (Mt CO2e)"

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

# One-hot encode object columns (e.g., Country Name) and align train/test columns
cat_cols = X_train.select_dtypes(include=["object"]).columns
if len(cat_cols) > 0:
    X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=False)
    X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=False)
    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (7812, 312)
Test shape: (1008, 312)


In [7]:
def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 200, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma': trial.suggest_float('gamma', 0.0, 5.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42,
        'n_jobs': -1,
        'tree_method': 'hist'
    }

    with mlflow.start_run(nested=True):
        model = XGBRegressor(**params)
        model.fit(X_train, y_train)

        y_hat = model.predict(X_test)
        mae = float(mean_absolute_error(y_test, y_hat))
        rmse = float(np.sqrt(mean_squared_error(y_test, y_hat)))
        r2 = float(r2_score(y_test, y_hat))

        mlflow.log_params(params)
        mlflow.log_metrics({'mae': mae, 'rmse': rmse, 'r2': r2})

    return rmse

In [8]:
mlflow.set_tracking_uri(os.path.join('..', 'mlruns'))
mlflow.set_experiment('xgboost_optuna_carbon_dioxide')

study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=15)

print("Best params:", study.best_trial.params)

[I 2026-04-04 14:41:26,479] A new study created in memory with name: no-name-c553920b-413e-4f94-8c5e-71572fd92a6e
[I 2026-04-04 14:41:28,679] Trial 0 finished with value: 813.0397363016306 and parameters: {'n_estimators': 419, 'max_depth': 9, 'learning_rate': 0.10568482928348323, 'subsample': 0.9099541596099563, 'colsample_bytree': 0.6160099355250375, 'min_child_weight': 1, 'gamma': 1.5105241835321648, 'reg_alpha': 3.520778468446729e-08, 'reg_lambda': 6.05408075950458e-07}. Best is trial 0 with value: 813.0397363016306.
[I 2026-04-04 14:41:31,494] Trial 1 finished with value: 621.2906408492004 and parameters: {'n_estimators': 955, 'max_depth': 5, 'learning_rate': 0.05407411748287727, 'subsample': 0.95158629158214, 'colsample_bytree': 0.6062712567254456, 'min_child_weight': 3, 'gamma': 1.6768025648040612, 'reg_alpha': 0.008020451905406347, 'reg_lambda': 0.498304843820502}. Best is trial 1 with value: 621.2906408492004.
[I 2026-04-04 14:41:37,508] Trial 2 finished with value: 677.6634985

Best params: {'n_estimators': 619, 'max_depth': 3, 'learning_rate': 0.2683205429997831, 'subsample': 0.8261687443669896, 'colsample_bytree': 0.7931747417160686, 'min_child_weight': 10, 'gamma': 4.838323444679246, 'reg_alpha': 8.074956269308561, 'reg_lambda': 0.010356921029305435}


In [9]:
best_params = study.best_trial.params
best_model= XGBRegressor(**best_params)
best_model.fit(X_train, y_train)

y_hat = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_hat)
rmse = np.sqrt(mean_squared_error(y_test, y_hat))
r2 = r2_score(y_test, y_hat)

print("Final tuned model performance")
print(f"MAE:  {mae}")
print(f"RMSE: {rmse}")
print(f"R²:   {r2}")

Final tuned model performance
MAE:  192.22576819286888
RMSE: 701.1980288742349
R²:   0.9780806607979075
